# ReCiPe 2016 Midpoint (H) cross-check

Characterised level only. No ReCiPe normalisation, no ReCiPe weighting: the screening rule
that selected the three reporting categories is an EF 3.1 procedure and is not repeated here.

The comparison is built on **ratios taken against Sc2 inside each method**, so the two methods
stay comparable where the unit differs (kg Sb eq against kg Cu eq). No quantity on this page is
expressed against Sc1.

Run the cells in order. The last cell writes `recipe_cross_check.xlsx` next to the notebook.

In [ ]:
# --- setup and pairing -------------------------------------------------------
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path

_try = ["Outputs/3_impact_assessment", "../Outputs/3_impact_assessment",
        "../../Outputs/3_impact_assessment"]
IA = next((Path(p) for p in _try if (Path(p) / "impact_EF31.csv").exists()), None)
assert IA is not None, "impact_assessment folder not found; set IA by hand"

ef = pd.read_csv(IA / "impact_EF31.csv");        ef["category"] = ef["category"].str.strip()
rc = pd.read_csv(IA / "impact_ReCiPe_mid.csv");  rc["category"] = rc["category"].str.strip()
ef["unit"] = ef["unit"].str.strip();             rc["unit"] = rc["unit"].str.strip()

# EF key, name as printed in the thesis, ReCiPe counterpart
PAIRS = [("Resource use minerals and metals", "Resource use, minerals and metals", "Mineral resource scarcity"),
         ("Climate change",                   "Climate change",                   "Global warming"),
         ("Eutrophication freshwater",        "Eutrophication, freshwater",       "Freshwater eutrophication")]

def row(df, cat):
    r = df.loc[df["category"] == cat]
    assert len(r) == 1, f"category not matched uniquely: {cat}"
    return r.iloc[0]
print('loaded:', len(ef), 'EF rows,', len(rc), 'ReCiPe categories')

## First result — the full category pairing

EF 3.1 and ReCiPe 2016 do not name the same categories and do not measure them in the same units. The pairing below is what makes any comparison possible, and printing it is what lets a reader check it.

**Reconstructed, needs your confirmation.** The 15-pair list you supplied last session lives in your notebook, not on disk here. The mapping below was rebuilt from the two category lists and reproduces all four of your own constraints exactly: 15 pairs, 4 sharing a unit, EF's *Eutrophication terrestrial* unpaired, and marine ecotoxicity, terrestrial ecotoxicity and ozone formation for terrestrial ecosystems unpaired on the ReCiPe side. Check it against your `PAIRS` before publishing.

In [ ]:
# --- the full category pairing, EF 3.1 against ReCiPe 2016 Midpoint (H) ------
ALL_PAIRS = [("Acidification",                                 "Terrestrial acidification"),
             ("Climate change",                                "Global warming"),
             ("Ecotoxicity freshwater",                        "Freshwater ecotoxicity"),
             ("Eutrophication freshwater",                     "Freshwater eutrophication"),
             ("Eutrophication marine",                         "Marine eutrophication"),
             ("Eutrophication terrestrial",                    None),
             ("Human toxicity cancer",                         "Human carcinogenic toxicity"),
             ("Human toxicity non-cancer",                     "Human non-carcinogenic toxicity"),
             ("Ionising radiation (human health)",             "Ionizing radiation"),
             ("Land use",                                      "Land use"),
             ("Ozone depletion",                               "Stratospheric ozone depletion"),
             ("Particulate matter",                            "Fine particulate matter formation"),
             ("Photochemical ozone formation (human health)",  "Ozone formation, Human health"),
             ("Resource use fossils",                          "Fossil resource scarcity"),
             ("Resource use minerals and metals",              "Mineral resource scarcity"),
             ("Water use",                                     "Water consumption")]

REPORTED = [k for k, _, _ in PAIRS]          # the three categories section 4.1 reports
efu, rcu = ef.set_index("category")["unit"], rc.set_index("category")["unit"]

rows = []
for a, b in ALL_PAIRS:
    ue = str(efu[a]).strip()
    ur = str(rcu[b]).strip() if b else ""
    rows.append({"EF 3.1 category": a, "EF unit": ue,
                 "ReCiPe counterpart": b if b else "no counterpart",
                 "ReCiPe unit": ur,
                 "same unit": ("yes" if ue == ur else "no") if b else "",
                 "reported in this study": "yes" if a in REPORTED else ""})
pairing = pd.DataFrame(rows)

n_pairs = sum(1 for _, b in ALL_PAIRS if b)
n_same  = int((pairing["same unit"] == "yes").sum())
unpaired_rc = sorted(set(rc["category"]) - {b for _, b in ALL_PAIRS if b})
unpaired_ef = [a for a, b in ALL_PAIRS if not b]

# the constraints the author stated for his own PAIRS list, checked, not assumed
assert n_pairs == 15,                 f"expected 15 pairs, built {n_pairs}"
assert n_same == 4,                   f"expected 4 shared units, found {n_same}"
assert unpaired_ef == ["Eutrophication terrestrial"], unpaired_ef
assert unpaired_rc == ["Marine ecotoxicity", "Ozone formation, Terrestrial ecosystems",
                       "Terrestrial ecotoxicity"], unpaired_rc
print(f"{n_pairs} pairs, {n_same} of them sharing a unit")
print("EF category with no counterpart:", unpaired_ef)
print("ReCiPe categories unpaired:", unpaired_rc)
pairing

In [ ]:
# --- does sharing a unit predict agreement? checked across all fifteen pairs --
chk = []
for a, b in ALL_PAIRS:
    if not b:
        continue
    e = ef.loc[ef["category"] == a].iloc[0]; r = rc.loc[rc["category"] == b].iloc[0]
    e2, e4 = float(e["sc2_saving"]), float(e["sc4_saving"])
    r2, r4 = float(r["sc2_saving"]), float(r["sc4_saving"])
    chk.append({"EF 3.1 category": a, "same unit": "yes" if str(e["unit"]).strip() == str(r["unit"]).strip() else "no",
                "EF Sc4 / Sc2": e4 / e2, "ReCiPe Sc4 / Sc2": r4 / r2,
                "absolute difference": abs(r4 / r2 - e4 / e2),
                "reported": "yes" if a in REPORTED else ""})
unit_check = pd.DataFrame(chk).sort_values("absolute difference", ascending=False)
print(unit_check.to_string(index=False, formatters={c: "{:.3f}".format for c in
      ["EF Sc4 / Sc2", "ReCiPe Sc4 / Sc2", "absolute difference"]}))
print()
print("Sharing a unit does NOT predict agreement across all fifteen pairs.")
print("  water use and water consumption differ in unit and agree to 0.000")
print("  ozone depletion shares its unit and differs by 0.241")
print("Within the three reported categories the two do coincide.")

## Second result — where the paired categories sit in ReCiPe's own ranking

ReCiPe 2016 Midpoint carries **normalisation factors but no midpoint weights**, so this ranking is normalised only. The EF 3.1 ranking in the screening block is normalised **and** weighted. The two are therefore not the same operation and must not be presented as equivalent rankings.

The three counterparts identified above are highlighted, so the reader can locate them without re-deriving the pairing.

What this ranking demonstrates is the size of the normalisation reference, not the size of the burden. Mineral resource scarcity is normalised against 1.2 x 10^5 kg Cu eq while the ecotoxicity categories are normalised against references of order 10^1, which is what places it last. This is the reason ReCiPe normalisation is not used to select categories in this study.

In [ ]:
# --- ReCiPe midpoint ranking, normalised only --------------------------------
scr = pd.read_csv(IA / "impact_screening_ReCiPe_mid.csv")
scr["category"] = scr["category"].str.strip()
scr = scr.sort_values("share_pct", ascending=False).reset_index(drop=True)

# the normalisation reference behind each share, derived: raw Sc1 / normalised person-equivalents
raw = rc.set_index("category")["sc1"].astype(float)
scr["Sc1 characterised"] = scr["category"].map(raw)
scr["normalisation reference"] = scr["Sc1 characterised"] / scr["weighted_pe"].astype(float)
scr["unit"] = scr["category"].map(rc.set_index("category")["unit"])

RC_SET = scr.loc[scr["in_reporting_set"].astype(bool), "category"].tolist()
EF_COUNTERPARTS = [rcn for _, _, rcn in PAIRS]

print("ReCiPe reporting set, smallest set reaching 80 percent cumulative:")
for c in RC_SET:
    r = scr.loc[scr["category"] == c].iloc[0]
    print(f"   {c:34s} {r['share_pct']:7.4f} %   cumulative {r['cum_pct']:7.4f} %")
print()
print("Where the three EF 3.1 reporting categories sit in this ranking:")
for c in EF_COUNTERPARTS:
    r = scr.loc[scr["category"] == c].iloc[0]
    rank = int(scr.index[scr["category"] == c][0]) + 1
    print(f"   {c:34s} rank {rank:2d} of 18   {r['share_pct']:.4f} %   "
          f"normalisation reference {r['normalisation reference']:.4g} {r['unit']}")

overlap = sorted(set(RC_SET) & set(EF_COUNTERPARTS))
print()
print("Categories in both reporting sets:", overlap if overlap else "none")
assert scr["share_pct"].sum() > 99.99, "shares do not close on 100"

In [ ]:
# --- figure, the ranking. Log axis: the shares span five orders of magnitude --
RC_C, EF_C, GREY = "#E69F00", "#0072B2", "#BBBBBB"

d = scr.iloc[::-1]                      # largest at the top of a horizontal bar chart
colors, edges = [], []
for c in d["category"]:
    if c in RC_SET:              colors.append(RC_C); edges.append("none")
    elif c in EF_COUNTERPARTS:   colors.append(EF_C); edges.append("none")
    else:                        colors.append(GREY); edges.append("none")

fig, ax = plt.subplots(figsize=(10.6, 6.6))
y = np.arange(len(d))
ax.barh(y, d["share_pct"].astype(float), color=colors, height=0.72)
ax.set_yticks(y); ax.set_yticklabels(d["category"], fontsize=9)
ax.set_xscale("log")
ax.set_xlim(1e-4, 200)
ax.set_xlabel("share of the normalised profile (%), logarithmic scale")
ax.grid(axis="x", lw=0.5, alpha=0.35); ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)

for yi, v in zip(y, d["share_pct"].astype(float)):
    ax.text(v * 1.25, yi, f"{v:.4f} %" if v < 1 else f"{v:.2f} %",
            va="center", fontsize=8.5)

ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, facecolor=RC_C),
                   plt.Rectangle((0, 0), 1, 1, facecolor=EF_C),
                   plt.Rectangle((0, 0), 1, 1, facecolor=GREY)],
          labels=["ReCiPe reporting set, 91.33 % cumulative",
                  "counterpart of an EF 3.1 reporting category",
                  "not in either set"],
          loc="lower right", frameon=False, fontsize=9)

ax.set_title("ReCiPe 2016 Midpoint (H), categories ranked by normalised contribution, Sc1 basis",
             fontsize=11.5, fontweight="bold", pad=12)
fig.text(0.5, -0.035,
         "Normalisation only: ReCiPe defines no midpoint weighting set, so this ranking is not the "
         "same operation as the normalised and weighted EF 3.1 ranking.",
         ha="center", fontsize=8.5, style="italic")
fig.tight_layout()

try:
    save(fig, "fig_recipe_midpoint_ranking")
except NameError:
    fig.savefig("fig_recipe_midpoint_ranking.png", dpi=300, bbox_inches="tight")
    fig.savefig("fig_recipe_midpoint_ranking.svg", bbox_inches="tight")
plt.show()

In [ ]:
# --- the mechanism, in one small table ---------------------------------------
mech = scr.loc[scr["category"].isin(EF_COUNTERPARTS + RC_SET),
               ["category", "unit", "Sc1 characterised", "normalisation reference",
                "weighted_pe", "share_pct"]].copy()
mech["set"] = np.where(mech["category"].isin(RC_SET), "ReCiPe reporting set",
                       "EF 3.1 counterpart")
mech = mech.rename(columns={"weighted_pe": "normalised (person-equivalents)",
                            "share_pct": "share of profile (%)"})
print(mech.to_string(index=False, formatters={
    "Sc1 characterised": "{:.4g}".format,
    "normalisation reference": "{:.4g}".format,
    "normalised (person-equivalents)": "{:.4g}".format,
    "share of profile (%)": "{:.4f}".format}))
ranking_table = scr[["category", "unit", "Sc1 characterised", "normalisation reference",
                     "weighted_pe", "share_pct", "cum_pct", "in_reporting_set"]]

In [ ]:
# --- TEST A, ordering. Ordinal, no reference scenario, no percentages --------
def monotonic(df, label):
    s2, s3, s4 = df["sc2_saving"].astype(float), df["sc3_saving"].astype(float), df["sc4_saving"].astype(float)
    ok = ((s2 < s3) & (s3 < s4))
    broken = df.loc[~ok, "category"].tolist()
    print(f"{label}: avoided impact increases Sc2 < Sc3 < Sc4 in {ok.sum()} of {len(df)} rows")
    if broken:
        print("   rows where it does not:", broken)
    return int(ok.sum()), len(df)

ef_ok, ef_n = monotonic(ef, "EF 3.1   (16 categories, 25 result rows)")
rc_ok, rc_n = monotonic(rc, "ReCiPe   (18 categories)")
assert ef_ok == ef_n and rc_ok == rc_n, "ordering claim would be false as written"

In [ ]:
# --- TEST B, ratios against Sc2. Dimensionless, comparable across methods ----
recs = []
for k, nice, rcn in PAIRS:
    for tag, r in (("EF 3.1", row(ef, k)), ("ReCiPe", row(rc, rcn))):
        s2, s3, s4 = float(r["sc2_saving"]), float(r["sc3_saving"]), float(r["sc4_saving"])
        recs.append({"Category pair": nice, "Counterpart": rcn, "Method": tag, "Unit": r["unit"],
                     "Sc3 / Sc2": s3 / s2, "Sc4 / Sc2": s4 / s2,
                     "Sc2 < Sc3 < Sc4": "yes" if s2 < s3 < s4 else "no"})
ratios = pd.DataFrame(recs)

# divergence between the two methods, in ratio points
div = (ratios.pivot_table(index="Category pair", columns="Method", values=["Sc3 / Sc2", "Sc4 / Sc2"])
       .assign(**{}))
delta = pd.DataFrame({
    "Sc3 / Sc2, ReCiPe minus EF": div[("Sc3 / Sc2", "ReCiPe")] - div[("Sc3 / Sc2", "EF 3.1")],
    "Sc4 / Sc2, ReCiPe minus EF": div[("Sc4 / Sc2", "ReCiPe")] - div[("Sc4 / Sc2", "EF 3.1")]})

print(ratios.to_string(index=False, formatters={"Sc3 / Sc2": "{:.3f}".format,
                                                "Sc4 / Sc2": "{:.3f}".format}))
print()
print(delta.to_string(formatters={c: "{:+.3f}".format for c in delta.columns}))

In [ ]:
# --- the independence check. Agreement is only evidence where the factors differ
lev = []
for k, nice, rcn in PAIRS:
    e, r = row(ef, k), row(rc, rcn)
    same_unit = e["unit"] == r["unit"]
    e2, r2 = float(e["sc2_saving"]), float(r["sc2_saving"])
    lev.append({"Category pair": nice, "shared unit": "yes" if same_unit else "no",
                "EF 3.1 Sc2 avoided": e2, "ReCiPe Sc2 avoided": r2,
                "relative difference (%)": (100.0 * abs(r2 - e2) / e2) if same_unit else np.nan})
levels = pd.DataFrame(lev)
print(levels.to_string(index=False, na_rep="not comparable",
                       formatters={"EF 3.1 Sc2 avoided": "{:.6g}".format,
                                   "ReCiPe Sc2 avoided": "{:.6g}".format,
                                   "relative difference (%)": "{:.4f}".format}))
print()
print("Where the two methods return nearly the same level, agreement reflects a shared")
print("characterisation factor and is not independent corroboration.")

In [ ]:
# --- optional, the gross finding under both methods --------------------------
def spread(r):
    v = np.array([float(r["sc1"]), float(r["sc2_gross"]), float(r["sc3_gross"]), float(r["sc4_gross"])])
    return 100.0 * (v.max() - v.min()) / v.min()

gross = pd.DataFrame([{"Category pair": nice,
                       "EF 3.1 gross spread (%)": spread(row(ef, k)),
                       "ReCiPe gross spread (%)": spread(row(rc, rcn))} for k, nice, rcn in PAIRS])
print(gross.to_string(index=False, formatters={c: "{:.4f}".format for c in gross.columns[1:]}))

allrc = rc.assign(spread=rc.apply(spread, axis=1)).sort_values("spread", ascending=False)
w = allrc.iloc[0]
print(f"\nLargest gross spread across the four scenarios in any ReCiPe category: "
      f"{w['spread']:.4f} % ({w['category']})")

In [ ]:
# --- figure, the six ratios. The only quantity comparable across the methods -
EF_C, RC_C = "#0072B2", "#E69F00"     # notebook palette, colour-vision-deficiency safe
labels = [nice for _, nice, _ in PAIRS]
short  = ["Minerals /\nmineral scarcity", "Climate change /\nglobal warming",
          "Freshwater\neutrophication"]

fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.4), sharey=True)
x = np.arange(len(PAIRS)); w = 0.36

for ax, col in zip(axes, ["Sc3 / Sc2", "Sc4 / Sc2"]):
    e = [ratios.loc[(ratios["Category pair"] == n) & (ratios["Method"] == "EF 3.1"), col].iloc[0] for n in labels]
    r = [ratios.loc[(ratios["Category pair"] == n) & (ratios["Method"] == "ReCiPe"), col].iloc[0] for n in labels]
    b1 = ax.bar(x - w/2, e, w, color=EF_C, label="EF 3.1", edgecolor="white")
    b2 = ax.bar(x + w/2, r, w, color=RC_C, label="ReCiPe 2016 Midpoint (H)", edgecolor="white")
    for bars in (b1, b2):
        ax.bar_label(bars, fmt="%.3f", fontsize=8.5, padding=2)
    ax.set_xticks(x); ax.set_xticklabels(short, fontsize=9)
    ax.set_title(col, fontsize=11, fontweight="bold")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", lw=0.5, alpha=0.35); ax.set_axisbelow(True)

axes[0].set_ylabel("avoided impact, times the Sc2 value")
axes[0].set_ylim(0, max(ratios["Sc4 / Sc2"]) * 1.18)
handles, lab = axes[0].get_legend_handles_labels()
fig.legend(handles, lab, loc="lower center", ncol=2, frameon=False, fontsize=9.5,
           bbox_to_anchor=(0.5, -0.04))
fig.suptitle("Avoided impact relative to Sc2 under two characterisation methods",
             fontsize=12.5, y=1.01)
fig.text(0.5, -0.13, "Ratios are taken inside each method, so the pairs remain comparable where the "
                     "unit differs (kg Sb eq against kg Cu eq).",
         ha="center", fontsize=8.5, style="italic")
fig.tight_layout()

try:
    save(fig, "fig_recipe_ratio_cross_check")      # notebook helper, cell 3
except NameError:
    fig.savefig("fig_recipe_ratio_cross_check.png", dpi=300, bbox_inches="tight")
    fig.savefig("fig_recipe_ratio_cross_check.svg", bbox_inches="tight")
plt.show()

In [ ]:
# --- export, so the tables reach Word without a decimal-separator conversion --
with pd.ExcelWriter("recipe_cross_check.xlsx") as xl:
    ranking_table.to_excel(xl, sheet_name="recipe_ranking", index=False)
    pairing.to_excel(xl, sheet_name="pairing",        index=False)
    ratios.to_excel(xl,  sheet_name="ratios_vs_Sc2",  index=False)
    levels.to_excel(xl,  sheet_name="independence",   index=False)
    gross.to_excel(xl,   sheet_name="gross_spread",   index=False)
    pd.DataFrame([{"check": "avoided increases Sc2 < Sc3 < Sc4",
                   "EF 3.1": f"{ef_ok} of {ef_n} rows",
                   "ReCiPe": f"{rc_ok} of {rc_n} categories"}]).to_excel(
                       xl, sheet_name="ordering", index=False)
print("written: recipe_cross_check.xlsx")